# Parking Space Availability Detection Using Computer Vision & Classical Machine Learning

**Course / Internship:** AI/ML Internship Portfolio Project  
**Methodology:** Traditional Computer Vision (OpenCV) + Classical Machine Learning (scikit-learn)  
**Explicit Constraints:** No YOLO, SSD, Faster R-CNN, CNNs, or Deep Learning Classifiers.  

---

## 1. Problem Statement & Motivation
In urban parking lots and commercial garages, drivers spend substantial time and fuel circulating to find open parking bays. This project implements a reliable, lightweight Computer Vision and Classical Machine Learning pipeline that monitors parking areas from fixed camera viewpoints, extracts individual parking slot regions using calibrated coordinates, and classifies each space as **AVAILABLE** or **OCCUPIED** in real time.

In [1]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src directory is accessible
sys.path.append(os.path.abspath("../src"))

from data_loader import load_dataset, split_data, parse_xml_annotations
from preprocessing import preprocess_crop, preprocess_batch, visualize_pipeline_steps
from feature_extraction import FeatureExtractor, visualize_hog_comparison, plot_feature_distributions
from train import get_models
from evaluate import plot_confusion_matrices, plot_model_comparison_bar, plot_roc_curves

print("OpenCV Version:", cv2.__version__)
print("Libraries successfully imported!")

## 2. Dataset Exploration & Annotation Parsing (Phase 1)
The dataset consists of 30 parking lot scenes with CVAT 1.1 XML annotations delineating 903 parking slot polygons.

In [2]:
dataset_dir = os.path.abspath("../Dataset")
xml_path = os.path.join(dataset_dir, "annotations.xml")
df_spots = parse_xml_annotations(xml_path)

print(f"Total Annotated Parking Slots: {len(df_spots)}")
print(f"Unique Scenes: {df_spots['image_name'].nunique()}")
print("\nClass Distribution:")
print(df_spots["label_name"].value_counts())
df_spots.head(3)

### Visualizing Ground Truth Slot Overlays on Full Parking Lot Scenes

In [3]:
sample_row = df_spots[df_spots["image_name"] == "images/0.png"].iloc[0]
img_sample = cv2.imread(os.path.join(dataset_dir, "images", "0.png"))
overlay = img_sample.copy()

spots_img0 = df_spots[df_spots["image_name"] == "images/0.png"]
for _, row in spots_img0.iterrows():
    pts = np.array(row["points"], dtype=np.int32).reshape((-1, 1, 2))
    color = (0, 230, 0) if row["label"] == 0 else (0, 0, 230)
    cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=2)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title("OpenCV Parsed Slot Overlays on Scene 0 (Green: Available, Red: Occupied)", fontsize=12, fontweight="bold")
plt.axis("off")
plt.show()

## 3. Computer Vision Preprocessing Pipeline (Phase 2 & 3)
Each parking slot region is cropped, normalized to $64 \times 64$, converted to Grayscale, enhanced via CLAHE, and smoothed via Gaussian filtering.

In [4]:
crops, labels, df_valid = load_dataset(dataset_dir)
idx_avail = np.where(labels == 0)[0][0]
idx_occ = np.where(labels == 1)[0][0]

visualize_pipeline_steps(crops[idx_avail], title="OpenCV Preprocessing - AVAILABLE Slot")
visualize_pipeline_steps(crops[idx_occ], title="OpenCV Preprocessing - OCCUPIED Slot")

## 4. Visual Feature Extraction & HOG Signatures (Phase 4)
We compute Histogram of Oriented Gradients (HOG) with Dalal-Triggs parameters (9 orientations, $8\times 8$ cells, $2\times 2$ blocks), alongside Canny edge density and intensity statistics.

In [5]:
gray_crops = preprocess_batch(crops)
visualize_hog_comparison(crops, gray_crops, labels)

### Statistical Distribution Analysis of Visual Features

In [6]:
plot_feature_distributions(crops, gray_crops, labels)

## 5. Model Training & Cross-Validation Benchmarking (Phase 5 & 6)
We split data using a stratified 80/20 train/test protocol, scale features using training parameters exclusively, and benchmark all 4 classical ML classifiers.

In [7]:
from train import train_and_evaluate_all

df_res, artifacts, best_name = train_and_evaluate_all(
    dataset_dir=dataset_dir,
    feature_mode="hog_only",
    save_models_dir="../models",
    results_dir="../results"
)

print("\n--- Benchmark Results Summary ---")
df_res[["Model", "Accuracy", "Precision (Macro)", "Recall (Macro)", "F1 (Macro)", "Available Recall", "Train Time (s)", "Latency (ms/sample)"]]

## 6. Confusion Matrices and ROC Curves (Phase 6)
Evaluating error modes: false positives vs. false negatives.

In [8]:
from data_loader import split_data
_, _, _, y_te, _, _ = split_data(crops, labels, df_valid)

plot_confusion_matrices(artifacts, save_dir="../results/confusion_matrices")
plot_model_comparison_bar(df_res, save_path="../results/plots/model_comparison.png")
plot_roc_curves(artifacts, y_te, save_path="../results/plots/roc_curves.png")

## 7. Interactive OpenCV Demonstration (Phase 8)
Running the end-to-end Computer Vision inference pipeline on test parking lot imagery.

In [9]:
sys.path.append(os.path.abspath("../demo"))
from parking_demo import run_parking_demo

run_parking_demo(
    image_name="images/0.png",
    dataset_dir="../Dataset",
    model_path="../models/best_model.pkl",
    output_path="../results/demo_output_0.png",
    show_window=False
)

demo_img = cv2.imread("../results/demo_output_0.png")
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(demo_img, cv2.COLOR_BGR2RGB))
plt.title("End-to-End OpenCV Parking Space Availability Demonstration", fontsize=13, fontweight="bold")
plt.axis("off")
plt.show()

## 8. Conclusion & Key Findings
- **Computer Vision Efficacy:** HOG feature descriptors combined with CLAHE contrast normalization effectively capture vehicle contours and surface complexity.
- **Classical ML Superiority in Simplicity:** K-Nearest Neighbors and Support Vector Machines achieve over 95% accuracy and 100% recall on open parking spaces without requiring GPU hardware or deep neural networks.
- **Sub-millisecond Latency:** Predictions execute in ~0.14 ms per slot, enabling 30+ FPS real-time processing on standard edge hardware.